In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://www.argentina.gob.ar/aaip/noticias"
print("✅ Scraper da AAIP (Argentina) pronto!")

✅ Scraper da AAIP (Argentina) pronto!


In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [5]:
# %%
def load_articles_from_db():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles_from_db()
display(df_db.head())
print(f"📦 Total no banco: {len(df_db)} registros")


,id,title,date,author,url,source
0,3090,EDPB brings clarity to data processing for sci...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
1,3091,Enhancing compliance and consistency: EDPB ado...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/enha...,EDPB
2,3092,EDPB annual report 2025: supporting stakeholde...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
3,3093,EDPB conference on cross-regulatory cooperatio...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
4,3094,EDPB and EDPS support strengthening EU’s cyber...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB


📦 Total no banco: 3700 registros


In [6]:
# %%
def montar_url(pagina):
    """Monta a URL paginada da AAIP (page=0 é a primeira)."""
    if pagina == 0:
        return BASE_URL
    return f"https://www.argentina.gob.ar/node/37875/noticias?page={pagina}"

def extrair_paragrafos(url):
    """Acessa a página da notícia e extrai os parágrafos principais."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        ps = soup.find_all("p")
        textos = [
            p.get_text(strip=True)
            for p in ps
            if len(p.get_text(strip=True).split()) > 10
        ]
        return textos[:5] if textos else ["NA"]
    except:
        return ["NA"]

In [7]:
# %%
noticias = []
TOTAL_PAGES = 5  # margem para cobrir todas as páginas (0-indexed) 36 cobre todas as paginas

for pagina in range(0, TOTAL_PAGES):
    url = montar_url(pagina)
    print(f"📄 Coletando página {pagina + 1}/{TOTAL_PAGES}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"⚠️ Erro HTTP {r.status_code} — parando.")
            break
    except Exception as e:
        print(f"⚠️ Erro de conexão: {e}")
        break

    soup = BeautifulSoup(r.text, "html.parser")

    noticia_tags = soup.select("a.panel[href*='/noticias/']")
    if not noticia_tags:
        noticia_tags = soup.select("a[href*='/noticias/']")

    if not noticia_tags:
        print("🏁 Sem mais resultados — fim da coleta.")
        break

    print(f"   {len(noticia_tags)} notícias encontradas")

    for a_tag in noticia_tags:
        h3 = a_tag.find("h3")
        if not h3:
            continue

        titulo = h3.get_text(strip=True)
        link = a_tag.get("href", "")
        if link.startswith("/"):
            link = f"https://www.argentina.gob.ar{link}"

        time_tag = a_tag.find("time")
        data_raw = time_tag.get_text(strip=True) if time_tag else "NA"

        # extrai parágrafos da notícia
        paragrafos = extrair_paragrafos(link)

        # acumula
        noticias.append({
            "titulo": titulo,
            "data": data_raw,
            "link": link,
            "paragrafos": " || ".join(paragrafos),
            "fonte": "AAIP Argentina"
        })

        # grava no banco
        insert_article(
            title=titulo,
            date=data_raw,
            author="AAIP",
            url=link,
            source="AAIP Argentina"
        )

    time.sleep(1)

print(f"\n✅ Total coletado: {len(noticias)} notícias")

df_aaip = pd.DataFrame(noticias)
display(df_aaip.head())

📄 Coletando página 1/5: https://www.argentina.gob.ar/aaip/noticias
   16 notícias encontradas
📄 Coletando página 2/5: https://www.argentina.gob.ar/node/37875/noticias?page=1
   16 notícias encontradas
📄 Coletando página 3/5: https://www.argentina.gob.ar/node/37875/noticias?page=2
   16 notícias encontradas
📄 Coletando página 4/5: https://www.argentina.gob.ar/node/37875/noticias?page=3
   16 notícias encontradas
📄 Coletando página 5/5: https://www.argentina.gob.ar/node/37875/noticias?page=4
   16 notícias encontradas

✅ Total coletado: 80 notícias


,titulo,data,link,paragrafos,fonte
0,La AAIP participó de la Cumbre Global de la IAPP,29 de abril de 2026,https://www.argentina.gob.ar/noticias/la-aaip-...,El encuentro anual convocó a autoridades y exp...,AAIP Argentina
1,La Red Iberoamericana de Protección de Datos r...,23 de abril de 2026,https://www.argentina.gob.ar/noticias/la-red-i...,La Red recolectará opiniones del mundo académi...,AAIP Argentina
2,La AAIP investiga presuntos incumplimientos a ...,13 de abril de 2026,https://www.argentina.gob.ar/noticias/la-aaip-...,La investigación se inició a partir de denunci...,AAIP Argentina
3,La AAIP encabezó la 65° reunión del Bureau del...,09 de abril de 2026,https://www.argentina.gob.ar/noticias/la-aaip-...,"La reunión convocó a los miembros del Bureau, ...",AAIP Argentina
4,Declaración conjunta de autoridades de protecc...,12 de marzo de 2026,https://www.argentina.gob.ar/noticias/declarac...,La iniciativa plantea principios fundamentales...,AAIP Argentina


In [8]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 3780 registros


,id,title,date,author,url,source
0,3090,EDPB brings clarity to data processing for sci...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
1,3091,Enhancing compliance and consistency: EDPB ado...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/enha...,EDPB
2,3092,EDPB annual report 2025: supporting stakeholde...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
3,3093,EDPB conference on cross-regulatory cooperatio...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
4,3094,EDPB and EDPS support strengthening EU’s cyber...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
5,3095,CEF 2026: EDPB launches coordinated enforcemen...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/cef-...,EDPB
6,3096,EDPB and EDPS support harmonisation of clinica...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
7,3097,Stakeholder event on political advertising: ag...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/stak...,EDPB
8,3098,Conference on cross-regulatory cooperation in ...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/conf...,EDPB
9,3099,AI-generated imagery and protection of privacy...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/ai-g...,EDPB


In [11]:
keywords = ['digital', 'internet', 'IA', 'tecnología', 'datos', 'privacidad',
            'protección', 'ciberseguridad', 'inteligencia artificial']
pattern = r'|'.join(keywords)

df_filt = df_aaip[
    df_aaip['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_aaip['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"{len(df_filt)} notícias filtradas (de {len(df_aaip)})")
display(df_filt.head())

80 notícias filtradas (de 80)


,titulo,data,link,paragrafos,fonte
0,La AAIP participó de la Cumbre Global de la IAPP,29 de abril de 2026,https://www.argentina.gob.ar/noticias/la-aaip-...,El encuentro anual convocó a autoridades y exp...,AAIP Argentina
1,La Red Iberoamericana de Protección de Datos r...,23 de abril de 2026,https://www.argentina.gob.ar/noticias/la-red-i...,La Red recolectará opiniones del mundo académi...,AAIP Argentina
2,La AAIP investiga presuntos incumplimientos a ...,13 de abril de 2026,https://www.argentina.gob.ar/noticias/la-aaip-...,La investigación se inició a partir de denunci...,AAIP Argentina
3,La AAIP encabezó la 65° reunión del Bureau del...,09 de abril de 2026,https://www.argentina.gob.ar/noticias/la-aaip-...,"La reunión convocó a los miembros del Bureau, ...",AAIP Argentina
4,Declaración conjunta de autoridades de protecc...,12 de marzo de 2026,https://www.argentina.gob.ar/noticias/declarac...,La iniciativa plantea principios fundamentales...,AAIP Argentina


In [12]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos")
        return

    # ------------------------------
    # Top 15
    # ------------------------------
    top15 = df.head(15).copy()
    top15['rank'] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x='rank',
        y='title',
        orientation='h',
        title='Top 15 Notícias – Internet Governance'
    )
    fig1.update_layout(height=600)
    fig1.show()

    # ------------------------------
    # Pizza por Fonte (BANCO)
    # ------------------------------
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # ------------------------------
    # Nuvem de Palavras
    # ------------------------------
    text = ' '.join(df['title'].astype(str)).lower()
    words = re.findall(r'\b\w{4,}\b', text)

    wc = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    wc.columns = ['palavra', 'freq']

    fig3 = px.treemap(
        wc,
        path=['palavra'],
        values='freq',
        title='Nuvem de Palavras – Títulos'
    )
    fig3.show()

In [13]:
plot_charts(df_db)